# Distributed Transactions & 2PC — the companion notebook

Every number in *System Design, Plainly — Season 2, Part 3* is recomputed here. Run all cells top to bottom; every code cell ends with asserts that must pass.

## 1. The availability arithmetic: every participant must be up

A 2PC transaction commits only if the coordinator *and* all $n$ participants are up. If each node is up 99.9% of the time, P(all up) = $0.999^n$ — and the downtime grows with every participant you add. This is the opposite of Part 2's quorum math, where more nodes *bought* availability.

In [ ]:
MIN_PER_YEAR = 525600.0

def all_up_downtime_min(n, p):
    """Minutes/year a 2PC txn cannot commit: any of n participants down."""
    return (1.0 - p ** n) * MIN_PER_YEAR

for p in (0.999, 0.9999):
    print(f'per-node uptime {p}:')
    for n in (1, 2, 3, 5):
        d = all_up_downtime_min(n, p)
        print(f'  n={n}: P(all up)={p**n:.6f}  down {d:.1f} min/yr ({d/60:.1f} h/yr)')

# cross-check the figures cited in the post: 0.999^3 -> ~26 h/yr, 0.999^5 -> ~44 h/yr
assert abs(all_up_downtime_min(3, 0.999) / 60 - 26.3) < 0.2
assert abs(all_up_downtime_min(5, 0.999) / 60 - 43.8) < 0.2
assert abs(all_up_downtime_min(1, 0.999) / 60 - 8.76) < 0.05
print('OK: 3 participants at 99.9% costs 26 h/yr of commit-ability; 5 costs 44 h/yr.')

## 2. The blocking problem, quantified

The coordinator is only *vulnerable* for a sliver of each transaction: the window between the last YES vote and the decision log write (~2 ms). If it crashes inside that window, every participant that voted YES is stuck in doubt, holding its locks, until the coordinator recovers. Expected in-doubt transactions per year = txn rate x window (for a coordinator with 1-year MTBF). Monte Carlo over 10,000 simulated years checks the arithmetic.

In [ ]:
import math, random

SEC_PER_YEAR = 31_536_000.0

def expected_blocked_per_year(txn_rate, window_s, mtbf_years=1.0):
    return txn_rate * SEC_PER_YEAR * (window_s / (mtbf_years * SEC_PER_YEAR))

for w_ms in (2, 5, 20):
    print(f'window {w_ms} ms @ 10k txn/s, MTBF 1 yr: '
          f'{expected_blocked_per_year(10_000, w_ms/1000):.1f} in-doubt txns/yr')

assert abs(expected_blocked_per_year(10_000, 0.002) - 20.0) < 1e-9
assert abs(expected_blocked_per_year(10_000, 0.005) - 50.0) < 1e-9

# Monte Carlo: Poisson(1) coordinator crash per year, Poisson(rate*window)
# transactions caught in the window per crash.
def poisson(lam, rng):
    L, k, p = math.exp(-lam), 0, 1.0
    while True:
        p *= rng.random()
        if p <= L:
            return k
        k += 1

rng = random.Random(42)
YEARS = 10_000
counts = [sum(poisson(20.0, rng) for _ in range(poisson(1.0, rng)))
          for _ in range(YEARS)]
mc_mean = sum(counts) / YEARS
print(f'Monte Carlo mean in-doubt txns/yr: {mc_mean:.2f} (expect ~20); '
      f'p99 {sorted(counts)[int(0.99*YEARS)]}, max {max(counts)}')
assert 18.0 < mc_mean < 22.0
print('OK: ~20 transactions a year get stranded in doubt — each holding its rows '
      'hostage for the full coordinator recovery time.')

## 3. The hot-row ceiling

A 2PC transaction holds its locks from the first write until the decision lands — across the whole network round trip. So the maximum update rate on a single hot row is exactly $1 / \text{lock-hold}$. At 20 ms of hold that's 50 updates/s, no matter how many machines you own. And as arrivals approach that ceiling, queueing delay explodes (M/M/1: mean wait $= h / (1-\rho)$).

In [ ]:
def hotrow_cap(hold_s):
    return 1.0 / hold_s

for h_ms, label in ((3, 'same-DC 2PC'), (20, '20 ms hold'), (162, 'cross-region 2PC')):
    print(f'{label}: hold {h_ms} ms -> hot-row cap {hotrow_cap(h_ms/1000):.1f} updates/s')

assert abs(hotrow_cap(0.020) - 50.0) < 1e-9
assert abs(hotrow_cap(0.003) - 333.33) < 0.01
assert abs(hotrow_cap(0.162) - 6.17) < 0.01

def mm1_wait(h_s, rho):
    """Mean queueing wait for an M/M/1 queue at utilization rho."""
    return h_s / (1.0 - rho)

print(f'M/M/1 wait, h=20 ms: 50% load -> {mm1_wait(0.020, 0.5)*1000:.0f} ms; '
      f'90% load -> {mm1_wait(0.020, 0.9)*1000:.0f} ms (10x the hold itself)')
assert abs(mm1_wait(0.020, 0.9) - 0.200) < 1e-9
print('OK: a hot row under 2PC tops out at 50 updates/s with a 20 ms hold — '
      'and 90% of that ceiling costs 10x the hold in queueing.')

## 4. The commit-latency ladder

Classic 2PC pays two network round trips plus two forced log writes per commit (the participant's prepare vote, and the coordinator's decision). CockroachDB's parallel commits fold the decision into the first round: 1 RTT + 1 write. Spanner pays the same 2 RTTs as classic 2PC, plus a commit wait equal to TrueTime's uncertainty (epsilon <= 7 ms) so timestamps respect real time.

In [ ]:
FSYNC_MS = 1.0   # one forced log write
EPSILON_MS = 7.0  # Spanner TrueTime uncertainty bound

def commit_latency(rtt_ms, protocol):
    if protocol == 'classic 2PC':
        return 2 * rtt_ms + 2 * FSYNC_MS
    if protocol == 'parallel commits':
        return 1 * rtt_ms + 1 * FSYNC_MS
    if protocol == 'Spanner-style':
        return 2 * rtt_ms + EPSILON_MS
    raise ValueError(protocol)

for name, rtt in (('Same datacenter', 0.5), ('Cross-AZ', 2.0), ('Cross-region', 80.0)):
    row = {p: round(commit_latency(rtt, p), 1)
           for p in ('classic 2PC', 'parallel commits', 'Spanner-style')}
    print(f'{name:16s} {row}')

assert abs(commit_latency(80.0, 'classic 2PC') - 162.0) < 1e-9
assert abs(commit_latency(80.0, 'parallel commits') - 81.0) < 1e-9
assert abs(commit_latency(0.5, 'classic 2PC') - 3.0) < 1e-9
print('OK: cross-region, classic 2PC costs 162 ms per commit; parallel commits '
      'halves it to 81 ms. Same-datacenter, the whole debate is 3 ms vs 1.5 ms.')

## 5. Sagas vs 2PC: what failure actually costs

A saga never blocks: a failed step is undone by a compensating action, and locks were already released. Its failure mode is different — a compensation that itself fails leaves the saga *unresolved*, needing a human. Compare that with 2PC, where a single participant being down aborts the transaction outright.

In [ ]:
steps, step_fail, comp_fail = 5, 0.01, 0.01
p_comp_needed = 1 - (1 - step_fail) ** steps   # at least one step fails
p_unresolved = p_comp_needed * comp_fail       # ...and its compensation fails too
p_2pc_abort = 1 - 0.999 ** 3                   # one of 3 participants down @ 99.9%

print(f'saga: P(compensation needed) = {p_comp_needed:.4f} ({p_comp_needed*100:.1f}% of sagas)')
print(f'saga: P(unresolved, needs a human) = {p_unresolved:.5f} (~1 in {1/p_unresolved:,.0f})')
print(f'2PC:  P(abort from a participant being down) = {p_2pc_abort:.4f} '
      f'({p_2pc_abort*100:.2f}% of txns)')

assert abs(p_comp_needed - 0.0490) < 0.0001
assert abs(p_unresolved - 0.00049) < 0.00001
assert abs(p_2pc_abort - 0.002997) < 1e-5
print('OK: 2PC fails loudly and rarely (0.30% aborts at 99.9% nodes); a saga fails '
      'quietly and needs compensations 4.9% of the time, with 1 in ~2,040 needing a human.')